# Talk to the deployed agent

Select this repository's `.venv` Python kernel, then run the cells in order. This notebook calls the existing App Service; it does not deploy resources or perform Agent 365 onboarding.

Configuration comes from `.local/deployment.json`, or `.local/http-auth.json` if deployment verification has not yet produced a readiness artifact. Public Entra mode uses your Azure CLI sign-in and the **HTTP API scope**, not the Agent 365 provisioning client. CIDR-only deployments still require access from an allowed network.

**Each message is a separate, single-turn model request and may incur charges.** No chat history is retained by the sample. Tokens stay in memory and are never printed. Clear notebook outputs before saving or sharing a conversation.

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlsplit

import requests
from azure.core.exceptions import ClientAuthenticationError
from azure.identity import AzureCliCredential

AGENT_URL = ""  # Optional HTTPS URL override; otherwise discover the deployed hostname.

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
deployment_path = root / ".local" / "deployment.json"
auth_path = root / ".local" / "http-auth.json"
deployment = (
    json.loads(deployment_path.read_text(encoding="utf-8-sig")) if deployment_path.is_file() else {}
)
http_auth = json.loads(auth_path.read_text(encoding="utf-8-sig")) if auth_path.is_file() else {}
if not deployment and not http_auth:
    raise RuntimeError("Run deployment/HTTP authentication setup first; no .local state found.")
if deployment and http_auth:
    if deployment["webAppResourceId"].lower() != http_auth["webAppResourceId"].lower():
        raise RuntimeError("The saved deployment and HTTP authentication target different apps.")

tenant_id = deployment.get("tenantId") or http_auth["tenantId"]
client_id = deployment.get("httpAuthClientId") or http_auth.get("clientId")
api_scope = f"api://{client_id}/.default" if client_id else None
credential = AzureCliCredential(tenant_id=tenant_id)


def auth_headers(scope):
    if scope is None:
        return {}
    try:
        token = credential.get_token(scope).token
    except ClientAuthenticationError:
        raise RuntimeError(
            f"Sign in from a terminal: az login --tenant {tenant_id} --scope {scope}"
        ) from None
    return {"Authorization": f"Bearer {token}"}


agent_url = AGENT_URL.strip().rstrip("/")
if not agent_url and deployment.get("hostname"):
    agent_url = f"https://{deployment['hostname']}"
if not agent_url:
    site_id = http_auth["webAppResourceId"]
    site = requests.get(
        f"https://management.azure.com{site_id}?api-version=2025-03-01",
        headers=auth_headers("https://management.azure.com/.default"),
        timeout=30,
        allow_redirects=False,
    )
    if site.status_code != 200:
        raise RuntimeError(
            f"Hostname lookup failed (HTTP {site.status_code}). Set AGENT_URL above instead."
        )
    agent_url = f"https://{site.json()['properties']['defaultHostName']}"
url = urlsplit(agent_url)
if url.scheme != "https" or not url.hostname or url.username or url.password or url.query:
    raise ValueError("AGENT_URL must be an HTTPS URL without credentials or a query string.")
if url.path not in ("", "/") or url.fragment:
    raise ValueError("AGENT_URL must contain only the app's origin, not a route or fragment.")
print("Agent:", agent_url)
print("Authentication:", "Microsoft Entra / Azure CLI" if api_scope else "Network allowlist")

## Check the app

HTTP 200 confirms the app is reachable and accepts your token. It does **not** prove model access; sending a message below checks that separately. If sign-in is needed, run the printed `az login` command in a terminal and rerun the cell.

In [ ]:
health = requests.get(
    f"{agent_url}/health",
    headers=auth_headers(api_scope),
    timeout=30,
    allow_redirects=False,
)
print(f"Health: HTTP {health.status_code}")

In [ ]:
def talk(message):
    try:
        response = requests.post(
            f"{agent_url}/chat",
            headers=auth_headers(api_scope),
            json={"message": message},
            timeout=120,
            allow_redirects=False,
        )
    except requests.RequestException:
        raise RuntimeError("Could not reach the agent. Check its URL and network access.") from None
    if response.status_code == 401:
        raise RuntimeError(
            f"Sign in from a terminal: az login --tenant {tenant_id} --scope {api_scope}"
        )
    if response.status_code == 403:
        raise RuntimeError("Access denied. Check the network allowlist and API authorization.")
    if response.status_code == 502:
        raise RuntimeError(
            "The app returned HTTP 502. Model access/connectivity may have failed; "
            "check App Service diagnostics. A successful health check is not a model check."
        )
    if response.status_code != 200:
        raise RuntimeError(f"Chat failed (HTTP {response.status_code}); response body withheld.")
    return response.json()["reply"]

## Send a message

Edit `message` and rerun this cell to talk to the agent. Each call obtains a current API token; bearer tokens and error-response bodies are not displayed.

In [ ]:
message = "Hello! What can you help me with?"
print(talk(message))